# CCG 2025 SieveVar Workflow

This notebook shows how the Chen, Chen, and Gao (2025) simulation setup is expressed using ordinary OptTreat models plus CCG-specific configuration objects. The model classes live in `opttreat.models.Model1` through `opttreat.models.Model15`; paper tuning choices live in the replication configuration below.

## 1. Load OptTreat Models And Runner

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from opttreat.models import Model1, Model8, Model15, get_model
from opttreat.simulations.ccg2025 import run_ccg2025_sievevar as ccg

pd.set_option("display.max_columns", None)

## 2. CCG M1-M15 Configuration Table

The model family defines only DGP functions. The CCG paper-specific choices are stored in `ccg.ALL_CCG_DESIGNS`: target type, spline segment counts, theorem group, value scaling, and level-set bandwidth.

In [ ]:
spec_rows = []
for spec in ccg.ALL_CCG_DESIGNS:
    spec_rows.append(
        {
            "model": spec.model_name,
            "theorem": spec.theorem,
            "parameter_type": spec.parameter_type.value,
            "dim": spec.dim,
            "target_lower": spec.target_lower,
            "target_upper": spec.target_upper,
            "J_segments_c": spec.j_segments_c,
            "J_segments_t": spec.j_segments_t,
            "eps": spec.eps,
            "value_scale": spec.value_scale,
        }
    )

spec_table = pd.DataFrame(spec_rows)
spec_table

## 3. Tiny Smoke Run

This runs one representative known-welfare model, one unknown/common-welfare model, and the value model. The settings are intentionally tiny and are not meant to reproduce the paper tables.

In [ ]:
summary, draws = ccg.main()
summary

## 4. Paper-Size Run Command

The full paper check is better run from a terminal because it is compute-heavy. Edit the top-level constants in `run_ccg2025_sievevar.py` to switch from smoke settings to paper-size settings.

In [ ]:
paper_command = """
python -m opttreat.simulations.ccg2025.run_ccg2025_sievevar
"""
print(paper_command)

## 5. Compare To Reported CCG Values

If the full-rep CSV exists, this cell compares it to the reported `n=1500` CCG SieveVar rows. For M7, use the rerun with R-compatible pseudoinverse tolerance if available.

In [ ]:
reported = pd.DataFrame(
    [
        ("M1", "Model1", 0.3857, 0.0152, 0.0469, 0.0467, 0.0035, 0.9370),
        ("M2", "Model2", 0.2358, 0.0039, 0.0481, 0.0488, 0.0029, 0.9480),
        ("M3", "Model3", 0.5001, 0.0038, 0.0580, 0.0581, 0.0023, 0.9560),
        ("M4", "Model4", 0.1033, 0.0185, 0.0478, 0.0482, 0.0088, 0.9400),
        ("M5", "Model5", 0.0499, 0.0282, 0.0418, 0.0431, 0.0110, 0.9310),
        ("M6", "Model6", 0.2315, 0.0268, 0.0559, 0.0552, 0.0057, 0.9260),
        ("M7", "Model7", 0.1250, 0.0462, 0.0505, 0.0502, 0.0087, 0.8920),
        ("M8", "Model8", 0.3857, 0.0068, 0.0414, 0.0417, 0.0026, 0.9475),
        ("M9", "Model9", 0.2358, 0.0042, 0.0425, 0.0431, 0.0026, 0.9555),
        ("M10", "Model10", 0.5001, 0.0067, 0.0511, 0.0511, 0.0020, 0.9480),
        ("M11", "Model11", 0.1033, 0.0307, 0.0365, 0.0379, 0.0041, 0.9065),
        ("M12", "Model12", 0.0499, 0.0418, 0.0316, 0.0344, 0.0050, 0.8410),
        ("M13", "Model13", 0.2315, 0.0177, 0.0432, 0.0438, 0.0030, 0.9420),
        ("M14", "Model14", 0.1250, 0.0251, 0.0382, 0.0386, 0.0050, 0.9230),
        ("M15", "Model15", 3.1416, 0.0076, 0.0710, 0.0711, 0.0092, 0.9420),
    ],
    columns=["label", "model", "truth_r", "bias_r", "sd_r", "se_r", "sdse_r", "coverage_r"],
)

summary_path = Path("/private/tmp/opttreat_ccg2025_fullrep_n1500/ccg2025_sievevar_summary_paper.csv")
m7_path = Path("/private/tmp/opttreat_ccg2025_fullrep_m7_rcond/ccg2025_sievevar_summary_paper.csv")

if summary_path.exists():
    python_summary = pd.read_csv(summary_path)
    if m7_path.exists():
        python_summary = python_summary[python_summary["model"] != "Model7"]
        python_summary = pd.concat([python_summary, pd.read_csv(m7_path)], ignore_index=True)
    comparison = reported.merge(python_summary, on="model")
    comparison[["label", "truth_r", "W_true", "bias_r", "bias", "sd_r", "sd", "se_r", "se", "coverage_r", "coverage"]]
else:
    print(f"Run the paper command first; missing {summary_path}")

## 6. R-Compatible `pinv` Tolerance

The CCG R code uses `MASS::ginv`, whose default tolerance is `sqrt(machine epsilon)`. NumPy's `pinv` default tolerance is much smaller, so it can retain nearly unidentified spline directions that R would drop. The CCG replication configs set `pinv_rcond=sqrt(np.finfo(float).eps)` to match R's generalized-inverse behavior.